# Auswertung nach Mitarbeitern

## Finde Städte, in denen sowohl Mitarbeiter als auch Kunden wohnen.

![mi001](Screenshots/mi001.png)

In [41]:
SELECT
  DISTINCT(m.Stadt)
FROM Personal.Mitarbeiter AS m
JOIN Verkauf.Bestellung As b ON m.MitarbeiterID = b.MitarbeiterID
JOIN CR.Kunde AS k ON b.KundenNr = k.KundenNr
WHERE m.Stadt = k.Stadt

(5 rows affected)

Stadt   
--------
Kirkland
London  
Redmond 
Seattle 
Tacoma  
(5 rows)

Total execution time: 00:00:00.012

## Wie viele verschiedene Kunden hatten die einzelnen Mitarbeiter im Jahr 2018?

Zeige MitarbeiterID, Nachname, Vorname und Anzahl versch. Kunden 2018. Sortiere nach MitarbeiterID

![mi002](Screenshots/mi002.png)

In [67]:
SELECT
  m.MitarbeiterID,
  m.Nachname,
  m.Vorname,
  COUNT(DISTINCT(b.KundenNr)) AS [Anzahl versch. Kunden 2018]
FROM Personal.Mitarbeiter AS m
JOIN Verkauf.Bestellung AS b ON m.MitarbeiterID = b.MitarbeiterID AND YEAR(b.Bestelldatum) = 2018
GROUP BY m.MitarbeiterID, m.Nachname, m.Vorname
ORDER BY m.MitarbeiterID

(18 rows affected)

MitarbeiterID | Nachname  | Vorname  | Anzahl versch. Kunden 2018
--------------+-----------+----------+---------------------------
1             | Davolio   | Nancy    | 30                        
2             | Fuller    | Andrew   | 37                        
3             | Leverling | Janet    | 52                        
4             | Peacock   | Margaret | 39                        
5             | Buchanan  | Steven   | 35                        
6             | Suyama    | Michael  | 44                        
7             | King      | Robert   | 37                        
8             | Callahan  | Laura    | 42                        
9             | Dodsworth | Anne     | 37                        
10            | Davis     | Sara     | 37                        
11            | Funk      | Don      | 39                        
12            | Lew       | Judy     | 36                        
13            | Peled     | Yael     | 33               

## Berechne den Umsatz je Mitarbeiter für das vierte Quartel 2018

Zeige MitarbeiterID, Nachname, Vorname und Umsatz Q4/2018. Sortiere nach Nachname und Vorname aufsteigend. Formatiere den Umsatz als Währung.

![mi003](Screenshots/mi003.png)

In [6]:
-- Lösung zeigt nur Mitarbeiter, die Umsatz hatten
SELECT
  m.MitarbeiterID,
  m.Nachname,
  m.Vorname,
  FORMAT(SUM(bd.Einzelpreis * bd.Bestellmenge * (1 - bd.Rabatt)), 'C', 'de-DE') AS [Umsatz Q4/2018]
FROM Personal.Mitarbeiter AS m
JOIN Verkauf.Bestellung AS b ON m.MitarbeiterID = b.MitarbeiterID
JOIN Verkauf.BestellDetail AS bd ON b.BestellungID = bd.BestellungID
WHERE b.bestelldatum >= '20181001' AND b.bestelldatum < '20190101' -- AND b.Status = 1
GROUP BY m.MitarbeiterID, m.Nachname, m.Vorname
ORDER BY m.Nachname, m.Vorname

(18 rows affected)

MitarbeiterID | Nachname  | Vorname  | Umsatz Q4/2018
--------------+-----------+----------+---------------
5             | Buchanan  | Steven   | 13.308,06 €   
8             | Callahan  | Laura    | 124.131,17 €  
17            | Cameron   | Maria    | 52.205,99 €   
10            | Davis     | Sara     | 37.058,40 €   
1             | Davolio   | Nancy    | 77.037,25 €   
9             | Dodsworth | Anne     | 73.237,50 €   
18            | Doyle     | Patricia | 78.521,97 €   
2             | Fuller    | Andrew   | 107.440,41 €  
11            | Funk      | Don      | 58.945,92 €   
7             | King      | Robert   | 171.459,59 €  
16            | King      | Russell  | 118.880,66 €  
3             | Leverling | Janet    | 89.804,66 €   
12            | Lew       | Judy     | 203.148,96 €  
14            | Mortensen | Sven     | 176.536,41 €  
4             | Peacock   | Margaret | 127.141,92 €  
13            | Peled     | Yael     | 83.359,30 €   
15      

In [8]:
-- Lösung zeigt auch Mitarbeiter ohne Umsatz
SELECT
  m.MitarbeiterID,
  m.Nachname,
  m.Vorname,
  FORMAT(ISNULL(SUM(bd.Einzelpreis * bd.Bestellmenge * (1 - bd.Rabatt)), 0), 'C', 'de-DE') AS [Umsatz Q4/2018]
FROM Personal.Mitarbeiter AS m
LEFT JOIN Verkauf.Bestellung AS b
    ON m.MitarbeiterID = b.MitarbeiterID
    AND b.Bestelldatum >= '20181001' AND b.Bestelldatum < '20190101'
LEFT JOIN Verkauf.BestellDetail AS bd ON b.BestellungID = bd.BestellungID
GROUP BY m.MitarbeiterID, m.Nachname, m.Vorname
ORDER BY m.Nachname, m.Vorname

Warnung: Ein NULL-Wert wird durch einen Aggregat- oder sonstigen SET-Vorgang gelöscht.
(18 rows affected)

MitarbeiterID | Nachname  | Vorname  | Umsatz Q4/2018
--------------+-----------+----------+---------------
5             | Buchanan  | Steven   | 13.308,06 €   
8             | Callahan  | Laura    | 124.131,17 €  
17            | Cameron   | Maria    | 52.205,99 €   
10            | Davis     | Sara     | 37.058,40 €   
1             | Davolio   | Nancy    | 77.037,25 €   
9             | Dodsworth | Anne     | 73.237,50 €   
18            | Doyle     | Patricia | 78.521,97 €   
2             | Fuller    | Andrew   | 107.440,41 €  
11            | Funk      | Don      | 58.945,92 €   
7             | King      | Robert   | 171.459,59 €  
16            | King      | Russell  | 118.880,66 €  
3             | Leverling | Janet    | 89.804,66 €   
12            | Lew       | Judy     | 203.148,96 €  
14            | Mortensen | Sven     | 176.536,41 €  
4             | Peacock   | M

## Berechne die Provision der Mitarbeiter für das vierte Quartal 2018 wie folgt:

- ab 150 Tsd. — 25%
- ab 100 Tsd. — 20%
- ab   50 Tsd. — 10%
- darunter:    —   5%

Zeige: MitarbeiterID, Nachname, Vorname, Umsatz Q4-2018, Provision % und Provision EUR.

Sortiere nach MitarbeiterID aufsteigend.

![mi005](Screenshots/mi005.png)

In [10]:
WITH Umsatz AS (
    SELECT
      m.MitarbeiterID,
      m.Nachname,
      m.Vorname,
      ISNULL(SUM(bd.Einzelpreis * bd.Bestellmenge * (1 - bd.Rabatt)), 0) AS UmsatzQ4
    FROM Personal.Mitarbeiter AS m
    LEFT JOIN Verkauf.Bestellung AS b
        ON m.MitarbeiterID = b.MitarbeiterID
        AND b.Bestelldatum >= '20181001' AND b.Bestelldatum < '20190101'
    LEFT JOIN Verkauf.BestellDetail AS bd ON b.BestellungID = bd.BestellungID
    GROUP BY m.MitarbeiterID, m.Nachname, m.Vorname
)
SELECT
  MitarbeiterID,
  Nachname,
  Vorname,
  FORMAT(UmsatzQ4, 'C', 'de-DE') AS [Umsatz Q4/2018],
  CASE
    WHEN UmsatzQ4 >= 150000 THEN 25
    WHEN UmsatzQ4 >= 100000 THEN 20
    WHEN UmsatzQ4 >= 50000 THEN 10
    ELSE 5
  END AS [Provision in %],
  FORMAT(
    UmsatzQ4 *
    CASE
      WHEN UmsatzQ4 >= 150000 THEN 25
      WHEN UmsatzQ4 >= 100000 THEN 20
      WHEN UmsatzQ4 >= 50000 THEN 10
      ELSE 5
    END / 100.0
  , 'C', 'de-DE') AS [Provision in €]
FROM Umsatz
ORDER BY MitarbeiterID

Warnung: Ein NULL-Wert wird durch einen Aggregat- oder sonstigen SET-Vorgang gelöscht.
(18 rows affected)

MitarbeiterID | Nachname  | Vorname  | Umsatz Q4/2018 | Provision in % | Provision in €
--------------+-----------+----------+----------------+----------------+---------------
1             | Davolio   | Nancy    | 77.037,25 €    | 10             | 7.703,73 €    
2             | Fuller    | Andrew   | 107.440,41 €   | 20             | 21.488,08 €   
3             | Leverling | Janet    | 89.804,66 €    | 10             | 8.980,47 €    
4             | Peacock   | Margaret | 127.141,92 €   | 20             | 25.428,38 €   
5             | Buchanan  | Steven   | 13.308,06 €    | 5              | 665,40 €      
6             | Suyama    | Michael  | 153.163,20 €   | 25             | 38.290,80 €   
7             | King      | Robert   | 171.459,59 €   | 25             | 42.864,90 €   
8             | Callahan  | Laura    | 124.131,17 €   | 20             | 24.826,23 €   
9            

## Zeige die Umsätze für Sara Davis nach Jahr und Quartal

[Referenz: DATEPART](https://learn.microsoft.com/en-us/sql/t-sql/functions/datepart-transact-sql?view=sql-server-ver16)

![mi006](Screenshots/mi006.png)

In [119]:
SELECT
  CAST(DATEPART(YEAR, b.Bestelldatum) AS VARCHAR) + ' - Q' + CAST(DATEPART(QUARTER, b.Bestelldatum) AS VARCHAR) AS [Quartal],
  SUM(bd.Einzelpreis * bd.Bestellmenge * (1 - bd.Rabatt)) AS Umsatz
FROM Personal.Mitarbeiter AS m
JOIN Verkauf.Bestellung AS b ON m.MitarbeiterID = b.MitarbeiterID
JOIN Verkauf.BestellDetail AS bd ON b.BestellungID = bd.BestellungID
WHERE m.Nachname = 'Davis' AND m.Vorname = 'Sara'
GROUP BY DATEPART(YEAR, b.Bestelldatum), DATEPART(QUARTER, b.Bestelldatum)
ORDER BY DATEPART(YEAR, b.Bestelldatum), DATEPART(QUARTER, b.Bestelldatum)

(37 rows affected)

Quartal   | Umsatz    
----------+-----------
2010 - Q1 | 124401,94 
2010 - Q2 | 66057,14  
2010 - Q3 | 168478,65 
2010 - Q4 | 141856,015
2011 - Q1 | 98415,50  
2011 - Q2 | 77394,15  
2011 - Q3 | 174832,085
2011 - Q4 | 93946,05  
2012 - Q1 | 150156,00 
2012 - Q2 | 111263,305
2012 - Q3 | 155077,795
2012 - Q4 | 145923,05 
2013 - Q1 | 145034,505
2013 - Q2 | 123301,77 
2013 - Q3 | 28557,96  
2013 - Q4 | 86011,44  
2014 - Q1 | 112069,74 
2014 - Q2 | 25432,18  
2014 - Q3 | 63884,23  
2014 - Q4 | 109472,45 
2015 - Q1 | 57446,765 
2015 - Q2 | 197408,62 
2015 - Q3 | 61484,50  
2015 - Q4 | 108897,19 
2016 - Q1 | 80937,60  
2016 - Q2 | 113845,44 
2016 - Q3 | 88042,635 
2016 - Q4 | 105378,06 
2017 - Q1 | 52480,93  
2017 - Q2 | 149428,795
2017 - Q3 | 81085,92  
2017 - Q4 | 368565,58 
2018 - Q1 | 146276,10 
2018 - Q2 | 108540,535
2018 - Q3 | 102868,57 
2018 - Q4 | 37058,40  
2019 - Q1 | 44719,88  
(37 rows)

Total execution time: 00:00:00.016